In [ ]:
import os
import shutil
from pathlib import Path
from PIL import Image

# Extrair a pasta do dataset UFPR-ALPR na pasta ../datasets/. É lá que ocorrerá a conversão
SOURCE_DATASET = "../datasets/UFPR-ALPR dataset"
OUTPUT_DATASET = "../datasets/dataset_yolo"

CLASS_VEHICLE = 0
CLASS_PLATE = 1

splits = ["training", "validation", "testing"]

def yolo_bbox(x, y, w, h, img_w, img_h):
    x_center = (x + w / 2) / img_w
    y_center = (y + h / 2) / img_h
    w_norm = w / img_w
    h_norm = h / img_h

    return f"{x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"

def parse_annotation(txt_path):
    vehicle_box = None
    plate_box = None

    with open(txt_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()

        # vehicle bbox
        if line.startswith("position_vehicle:"):
            values = line.replace("position_vehicle:", "").strip().split()

            x = int(values[0])
            y = int(values[1])
            w = int(values[2])
            h = int(values[3])

            vehicle_box = (x, y, w, h)

        # plate polygon
        elif line.startswith("corners:"):
            corners_str = line.replace("corners:", "").strip()

            pts = corners_str.split()

            xs = []
            ys = []

            for pt in pts:
                px, py = pt.split(",")
                xs.append(int(px))
                ys.append(int(py))

            xmin = min(xs)
            xmax = max(xs)
            ymin = min(ys)
            ymax = max(ys)

            plate_box = (
                xmin,
                ymin,
                xmax - xmin,
                ymax - ymin
            )

    return vehicle_box, plate_box

# create output folders
for split in ["train", "val", "test"]:
    os.makedirs(f"{OUTPUT_DATASET}/{split}/images", exist_ok=True)
    os.makedirs(f"{OUTPUT_DATASET}/{split}/labels", exist_ok=True)

split_map = {
    "training": "train",
    "validation": "val",
    "testing": "test"
}

for split in splits:

    split_name = split_map[split]

    split_path = Path(SOURCE_DATASET) / split

    for track_dir in split_path.iterdir():

        if not track_dir.is_dir():
            continue

        for img_path in track_dir.glob("*.png"):

            txt_path = img_path.with_suffix(".txt")

            if not txt_path.exists():
                continue

            vehicle_box, plate_box = parse_annotation(txt_path)

            # img = Image.open(img_path)
            # img_w, img_h = img.size
            img_w = 1920
            img_h = 1080

            label_lines = []

            if vehicle_box:
                x, y, w, h = vehicle_box

                label_lines.append(
                    f"{CLASS_VEHICLE} "
                    + yolo_bbox(x, y, w, h, img_w, img_h)
                )

            if plate_box:
                x, y, w, h = plate_box

                label_lines.append(
                    f"{CLASS_PLATE} "
                    + yolo_bbox(x, y, w, h, img_w, img_h)
                )

            # output filenames
            new_name = f"{track_dir.name}_{img_path.stem}"

            out_img = (
                Path(OUTPUT_DATASET)
                / split_name
                / "images"
                / f"{new_name}.png"
            )

            out_label = (
                Path(OUTPUT_DATASET)
                / split_name
                / "labels"
                / f"{new_name}.txt"
            )

            shutil.copy(img_path, out_img)

            with open(out_label, "w") as f:
                f.write("\n".join(label_lines))

print("Conversion complete.")

Conversion complete.
